In [ ]:
import torch
import pickle as pkl
import json
from datetime import datetime
import os
import pandas as pd
import numpy as np

In [ ]:
from google.colab import drive
drive.mount('/absolute/path/to/')
base_path =  '/absolute/path/to/GETNext'
original_path = '/absolute/path/to/SafetyIsAllYouNeed/baselines/exports'

Mounted at /absolute/path/to/


In [ ]:
!git clone https://github.com/songyangme/GETNext.git
%cd GETNext

Cloning into 'GETNext'...
remote: Enumerating objects: 55, done.
remote: Counting objects: 100% (16/16), done.
remote: Compressing objects: 100% (5/5), done.
remote: Total 55 (delta 12), reused 11 (delta 11), pack-reused 39 (from 1)
Receiving objects: 100% (55/55), 6.69 MiB | 25.28 MiB/s, done.
Resolving deltas: 100% (18/18), done.
/absolute/path/to/GETNext


### Load original trajectories

In [ ]:
with open(os.path.join(original_path, 'train_trajectories.pickle'), 'rb') as f:
    train_trajs = pkl.load(f)

with open(os.path.join(original_path, 'validation_trajectories.pickle'), 'rb') as f:
    validation_trajs = pkl.load(f)

with open(os.path.join(original_path, 'test_trajectories.pickle'), 'rb') as f:
    test_trajs = pkl.load(f)

train_df = pd.read_csv(os.path.join(original_path, 'train_checkins.csv'))
validation_df = pd.read_csv(os.path.join(original_path, 'validation_checkins.csv'))
test_df = pd.read_csv(os.path.join(original_path, 'test_checkins.csv'))

In [ ]:
def make_getnext_csv(trajectories: list[pd.DataFrame], out_path: str):
    """
    Given a list of length-20 DataFrames (columns at least:
      user_id, poi_id, poi_category_id, poi_category_name,
      latitude, longitude, timezone_offset, utc_time, local_time
    ), produce one big CSV in GETNext format.
    """
    rows = []
    for traj_idx, traj in enumerate(trajectories):
        # traj is a DataFrame of exactly 20 rows, sorted by local_time
        user = traj.user_id.iloc[0]
        window_id = f"{user}_{traj_idx}"
        # precompute the day of first check-in to get day_shift
        first_day = traj.local_time.dt.date.iloc[0]
        for pos, row in enumerate(traj.itertuples(index=False), start=1):
            # pos: 1..20
            # fields in 'row':
            #  row.user_id, row.poi_id, row.poi_category_id, row.poi_category_name,
            #  row.latitude, row.longitude,
            #  row.timezone_offset, row.utc_time, row.local_time
            #
            # build the GETNext row:
            day_of_week = row.local_time.weekday()  # Monday=0
            # fraction of day past midnight
            tod = row.local_time.hour * 3600 + row.local_time.minute*60 + row.local_time.second
            norm_in_day = tod / (24*3600)
            # how many days since first_day
            shift = (row.local_time.date() - first_day).days
            # relative position in window (0-1)
            norm_rel = (pos - 1) / (len(traj)-1)  # 0.0 at first, 1.0 at last

            rows.append({
                "user_id":           user,
                "POI_id":            row.poi_id,
                "POI_catid":         row.poi_category_id,
                "POI_catid_code":    row.poi_category_id_code,   # if you pre-map that
                "POI_catname":       row.poi_category_name,
                "latitude":          row.latitude,
                "longitude":         row.longitude,
                "timezone":          row.timezone_offset,
                "UTC_time":          row.utc_time,
                "local_time":        row.local_time,
                "day_of_week":       day_of_week,
                "norm_in_day_time":  norm_in_day,
                "trajectory_id":     window_id,
                "norm_day_shift":    shift,
                "norm_relative_time":norm_rel,
            })

    df_out = pd.DataFrame(rows)
    # GETNext wants **no header** and **tab**-separated
    # df_out.to_csv(out_path, sep="\t", header=False, index=False)
    df_out.to_csv(out_path, index=False, sep=',', header=[
  "user_id","POI_id","POI_catid","POI_catid_code","POI_catname",
  "latitude","longitude","timezone","UTC_time","local_time",
  "day_of_week","norm_in_day_time","trajectory_id",
  "norm_day_shift","norm_relative_time"
])


In [ ]:
unique_cats = train_df['poi_category_id'].unique()
unique_cats.sort()
code_map = {cat: idx for idx, cat in enumerate(unique_cats)}


def add_category_code_to_trajs(trajs, code_map):
    for traj in trajs:
        traj['poi_category_id_code'] = traj['poi_category_id'].map(code_map)
    return trajs


train_trajs = add_category_code_to_trajs(train_trajs, code_map)
val_trajs   = add_category_code_to_trajs(validation_trajs,   code_map)
test_trajs  = add_category_code_to_trajs(test_trajs,  code_map)

In [ ]:
make_getnext_csv(train_trajs, os.path.join(base_path, "dataset/NYC/NYC_train.csv"))
make_getnext_csv(val_trajs,   os.path.join(base_path, "dataset/NYC/NYC_val.csv"))
make_getnext_csv(test_trajs,  os.path.join(base_path, "dataset/NYC/NYC_test.csv"))

In [ ]:
# Make sure the directory exists
!mkdir -p dataset/NYC

# Copy your Drive-mounted CSVs into the local GETNext tree
!cp "/absolute/path/to/GETNext/dataset/NYC/NYC_train.csv" dataset/NYC/
!cp "/absolute/path/to/GETNext/dataset/NYC/NYC_val.csv"   dataset/NYC/
!cp "/absolute/path/to/GETNext/dataset/NYC/NYC_test.csv"  dataset/NYC/

In [ ]:
!python build_graph.py --data-train dataset/NYC/NYC_train.csv

Build global POI checkin graph -----------------------------------
100% 88/88 [00:06<00:00, 12.85it/s]


### Train Model

In [ ]:
!python train.py \
  --data-train dataset/NYC/NYC_train.csv \
  --data-val   dataset/NYC/NYC_val.csv \
  --time-units 48 \
  --time-feature norm_in_day_time \
  --poi-embed-dim 128 \
  --user-embed-dim 128 \
  --time-embed-dim 32 \
  --cat-embed-dim 32 \
  --node-attn-nhid 128 \
  --transformer-nhid 1024 \
  --transformer-nlayers 2 \
  --transformer-nhead 2 \
  --batch 8 \
  --epochs 30 \
  --name  nyc_colab_run

2025-05-22 09:48:15 Namespace(seed=42, device=device(type='cuda'), data_adj_mtx='dataset/NYC/graph_A.csv', data_node_feats='dataset/NYC/graph_X.csv', data_train='dataset/NYC/NYC_train.csv', data_val='dataset/NYC/NYC_test.csv', short_traj_thres=2, time_units=48, time_feature='norm_in_day_time', poi_embed_dim=128, user_embed_dim=128, gcn_dropout=0.3, gcn_nhid=[32, 64], transformer_nhid=1024, transformer_nlayers=2, transformer_nhead=2, transformer_dropout=0.3, time_embed_dim=32, cat_embed_dim=32, time_loss_weight=10, node_attn_nhid=128, batch=8, epochs=30, lr=0.001, lr_scheduler_factor=0.1, weight_decay=0.0005, save_weights=True, save_embeds=False, workers=0, project='runs/train', name='nyc_colab_run', exist_ok=False, no_cuda=False, mode='client', port=64973, feature1='checkin_cnt', feature2='poi_catid', feature3='latitude', feature4='longitude', save_dir='runs/train/nyc_colab_run-2')
Loading POI graph...
2025-05-22 09:48:18 raw_X.shape: (4609, 4); Four features: checkin_cnt, poi_catid, l